# Clasificación de Piso en el Dataset UJIIndoorLoc

---

## Introducción

En este notebook se implementa un flujo completo de procesamiento y análisis para la clasificación del **piso** en un entorno interior utilizando el dataset **UJIIndoorLoc**. Este conjunto de datos contiene mediciones de señales WiFi recopiladas en distintas ubicaciones de un edificio, con información sobre coordenadas, piso, usuario, hora, entre otros.

En esta tarea nos enfocaremos en predecir el **piso** en el que se encuentra un dispositivo, considerando únicamente las muestras etiquetadas con valores válidos para dicha variable. Se tratará como un problema de clasificación multiclase (planta baja, primer piso, segundo piso).

## Objetivos

- **Cargar y explorar** el conjunto de datos UJIIndoorLoc.
- **Preparar** los datos seleccionando las características relevantes y el target (`FLOOR`).
- **Dividir** el dataset en entrenamiento y validación (80/20).
- **Entrenar y optimizar** clasificadores basados en seis algoritmos:
  - K-Nearest Neighbors (KNN)
  - Gaussian Naive Bayes
  - Regresión Logística
  - Árboles de Decisión
  - Support Vector Machines (SVM)
  - Random Forest
- **Seleccionar hiperparámetros óptimos** para cada modelo utilizando validación cruzada (5-fold), empleando estrategias como **Grid Search**, **Randomized Search**, o **Bayesian Optimization** según el algoritmo.
- **Comparar el desempeño** de los modelos sobre el conjunto de validación, usando métricas como *accuracy*, *precision*, *recall*, y *F1-score*.
- **Determinar el mejor clasificador** para esta tarea, junto con sus hiperparámetros óptimos.

Este ejercicio permite no solo evaluar la capacidad predictiva de distintos algoritmos clásicos de clasificación, sino también desarrollar buenas prácticas en validación de modelos y selección de hiperparámetros en contextos del mundo real.

---

## Descripción del Dataset

El dataset utilizado en este análisis es el **UJIIndoorLoc Dataset**, ampliamente utilizado para tareas de localización en interiores a partir de señales WiFi. Está disponible públicamente en la UCI Machine Learning Repository y ha sido recopilado en un entorno real de un edificio universitario.

Cada muestra corresponde a una observación realizada por un dispositivo móvil, donde se registran las intensidades de señal (RSSI) de más de 500 puntos de acceso WiFi disponibles en el entorno. Además, cada fila contiene información contextual como la ubicación real del dispositivo (coordenadas X e Y), el piso, el edificio, el identificador del usuario, y la marca temporal.

El objetivo en esta tarea es predecir el **piso** (`FLOOR`) en el que se encontraba el dispositivo en el momento de la medición, considerando únicamente las características numéricas provenientes de las señales WiFi.

### Estructura del dataset

- **Número de muestras**: ~20,000
- **Número de características**: 520
  - 520 columnas con valores de intensidad de señal WiFi (`WAP001` a `WAP520`)
- **Variable objetivo**: `FLOOR` (variable categórica con múltiples clases, usualmente entre 0 y 4)

### Columnas relevantes

- `WAP001`, `WAP002`, ..., `WAP520`: niveles de señal recibida desde cada punto de acceso WiFi (valores entre -104 y 0, o 100 si no se detectó).
- `FLOOR`: clase objetivo a predecir (nivel del edificio).
- (Otras columnas como `BUILDINGID`, `SPACEID`, `USERID`, `TIMESTAMP`, etc., pueden ser ignoradas o utilizadas en análisis complementarios).

### Contexto del problema

La localización en interiores es un problema complejo en el que tecnologías como el GPS no funcionan adecuadamente. Los sistemas basados en WiFi han demostrado ser una alternativa efectiva para estimar la ubicación de usuarios en edificios. Poder predecir automáticamente el piso en el que se encuentra una persona puede mejorar aplicaciones de navegación en interiores, accesibilidad, gestión de emergencias y servicios personalizados. Este tipo de problemas es típicamente abordado mediante algoritmos de clasificación multiclase.


### Estrategia de evaluación

En este análisis seguiremos una metodología rigurosa para garantizar la validez de los resultados:

1. **Dataset de entrenamiento**: Se utilizará exclusivamente para el desarrollo, entrenamiento y optimización de hiperparámetros de todos los modelos. Este conjunto será dividido internamente en subconjuntos de entrenamiento y validación (80/20) para la selección de hiperparámetros mediante validación cruzada.

2. **Dataset de prueba**: Se reservará únicamente para la **evaluación final** de los modelos ya optimizados. Este conjunto **no debe ser utilizado** durante el proceso de selección de hiperparámetros, ajuste de modelos o toma de decisiones sobre la arquitectura, ya que esto introduciría sesgo y comprometería la capacidad de generalización estimada.

3. **Validación cruzada**: Para la optimización de hiperparámetros se empleará validación cruzada 5-fold sobre el conjunto de entrenamiento, lo que permitirá una estimación robusta del rendimiento sin contaminar los datos de prueba.

Esta separación estricta entre datos de desarrollo y evaluación final es fundamental para obtener una estimación realista del rendimiento que los modelos tendrían en un escenario de producción con datos completamente nuevos.

---


## Paso 1: Cargar y explorar el dataset

**Instrucciones:**
- Descarga el dataset **UJIIndoorLoc** desde la UCI Machine Learning Repository o utiliza la versión proporcionada en el repositorio del curso (por ejemplo: `datasets\UJIIndoorLoc\trainingData.csv`).
- Carga el dataset utilizando `pandas`.
- Muestra las primeras filas del dataset utilizando `df.head()`.
- Imprime el número total de muestras (filas) y características (columnas).
- Verifica cuántas clases distintas hay en la variable objetivo `FLOOR` y cuántas muestras tiene cada clase (`df['FLOOR'].value_counts()`).


In [1]:
import pandas as pd

# Cargar dataset de entrenamiento
df = pd.read_csv('dataset/trainingData.csv')
print('Dataset cargado desde: dataset/trainingData.csv')

# 1) Primeras filas
display(df.head())

# 2) Número de muestras y características
print(f'Número de muestras (filas): {df.shape[0]}')
print(f'Número de características (columnas): {df.shape[1]}')

# 3) Clases de FLOOR y distribución
if 'FLOOR' not in df.columns:
    raise KeyError("La columna 'FLOOR' no existe en el dataset cargado.")

floor_counts = df['FLOOR'].value_counts().sort_index()
print(f'Número de clases distintas en FLOOR: {df["FLOOR"].nunique()}')
print('\nMuestras por clase (FLOOR):')
print(floor_counts)

Dataset cargado desde: dataset/trainingData.csv


,WAP001,WAP002,WAP003,WAP004,WAP005,WAP006,WAP007,WAP008,WAP009,WAP010,...,WAP520,LONGITUDE,LATITUDE,FLOOR,BUILDINGID,SPACEID,RELATIVEPOSITION,USERID,PHONEID,TIMESTAMP
0,100,100,100,100,100,100,100,100,100,100,...,100,-7541.2643,4.864921e+06,2,1,106,2,2,23,1371713733
1,100,100,100,100,100,100,100,100,100,100,...,100,-7536.6212,4.864934e+06,2,1,106,2,2,23,1371713691
2,100,100,100,100,100,100,100,-97,100,100,...,100,-7519.1524,4.864950e+06,2,1,103,2,2,23,1371714095
3,100,100,100,100,100,100,100,100,100,100,...,100,-7524.5704,4.864934e+06,2,1,102,2,2,23,1371713807
4,100,100,100,100,100,100,100,100,100,100,...,100,-7632.1436,4.864982e+06,0,0,122,2,11,13,1369909710


Número de muestras (filas): 19937
Número de características (columnas): 529
Número de clases distintas en FLOOR: 5

Muestras por clase (FLOOR):
FLOOR
0    4369
1    5002
2    4416
3    5048
4    1102
Name: count, dtype: int64


---

## Paso 2: Preparar los datos

**Instrucciones:**

- Elimina las columnas que no son relevantes para la tarea de clasificación del piso:
  - `LONGITUDE`, `LATITUDE`, `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP`
- Conserva únicamente:
  - Las columnas `WAP001` a `WAP520` como características (RSSI de puntos de acceso WiFi).
  - La columna `FLOOR` como variable objetivo.
- Verifica si existen valores atípicos o valores inválidos en las señales WiFi (por ejemplo: valores constantes como 100 o -110 que suelen indicar ausencia de señal).
- Separa el conjunto de datos en:
  - `X`: matriz de características (todas las columnas `WAP`)
  - `y`: vector objetivo (`FLOOR`)


In [2]:
# Columnas no relevantes para la clasificación de FLOOR
cols_to_drop = [
    'LONGITUDE', 'LATITUDE', 'SPACEID', 'RELATIVEPOSITION',
    'USERID', 'PHONEID', 'TIMESTAMP'
 ]

# Crear copia limpia del dataset
df_clean = df.drop(columns=cols_to_drop, errors='ignore').copy()

# Seleccionar columnas WAP como características
wap_cols = [c for c in df_clean.columns if c.startswith('WAP')]
if len(wap_cols) != 520:
    print(f'Aviso: se detectaron {len(wap_cols)} columnas WAP (esperadas: 520).')

# Verificar columna objetivo
if 'FLOOR' not in df_clean.columns:
    raise KeyError("La columna 'FLOOR' no existe después de la limpieza.")

# Revisar valores típicamente usados como ausencia de señal
total_wap_values = df_clean[wap_cols].size
count_100 = (df_clean[wap_cols] == 100).sum().sum()
count_m110 = (df_clean[wap_cols] == -110).sum().sum()

print(f'Total de valores en columnas WAP: {total_wap_values:,}')
print(f'Cantidad de valores 100 (sin señal): {count_100:,} ({(count_100/total_wap_values)*100:.2f}%)')
print(f'Cantidad de valores -110: {count_m110:,} ({(count_m110/total_wap_values)*100:.2f}%)')

# Separar X e y
X = df_clean[wap_cols].copy()
y = df_clean['FLOOR'].copy()

print(f'\nForma de X: {X.shape}')
print(f'Forma de y: {y.shape}')
print(f'Clases en y: {sorted(y.unique())}')

Total de valores en columnas WAP: 10,367,240
Cantidad de valores 100 (sin señal): 10,008,477 (96.54%)
Cantidad de valores -110: 0 (0.00%)

Forma de X: (19937, 520)
Forma de y: (19937,)
Clases en y: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


--- 

## Paso 3: Preprocesamiento de las señales WiFi

**Contexto:**

Las columnas `WAP001` a `WAP520` representan la intensidad de la señal (RSSI) recibida desde distintos puntos de acceso WiFi. Los valores típicos de RSSI están en una escala negativa, donde:

- Valores cercanos a **0 dBm** indican señal fuerte.
- Valores cercanos a **-100 dBm** indican señal débil o casi ausente.
- Un valor de **100** en este dataset representa una señal **no detectada**, es decir, el punto de acceso no fue visto por el dispositivo en ese instante.

**Instrucciones:**

- Para facilitar el procesamiento y tratar la ausencia de señal de forma coherente, se recomienda mapear todos los valores **100** a **-100**, que semánticamente representa *ausencia de señal detectable*.
- Esto unifica el rango de valores y evita que 100 (un valor artificial) afecte negativamente la escala de los algoritmos.

**Pasos sugeridos:**

- Reemplaza todos los valores `100` por `-100` en las columnas `WAP001` a `WAP520`:
  ```python
  X[X == 100] = -100


In [3]:
# Paso 3: reemplazar ausencia de señal codificada como 100 por -100
count_100_before = (X == 100).sum().sum()
print(f'Valores 100 antes del reemplazo: {count_100_before:,}')

X = X.copy()
X[X == 100] = -100

count_100_after = (X == 100).sum().sum()
count_m100_after = (X == -100).sum().sum()

print(f'Valores 100 después del reemplazo: {count_100_after:,}')
print(f'Valores -100 después del reemplazo: {count_m100_after:,}')

# Verificación rápida del rango luego del preprocesamiento
print(f'Valor mínimo en X: {X.min().min()}')
print(f'Valor máximo en X: {X.max().max()}')

Valores 100 antes del reemplazo: 10,008,477
Valores 100 después del reemplazo: 0
Valores -100 después del reemplazo: 10,008,716
Valor mínimo en X: -104
Valor máximo en X: 0


--- 

## Paso 4: Entrenamiento y optimización de hiperparámetros

**Objetivo:**

Entrenar y comparar distintos clasificadores para predecir correctamente el piso (`FLOOR`) y encontrar los mejores hiperparámetros para cada uno mediante validación cruzada.

**Clasificadores a evaluar:**

- K-Nearest Neighbors (KNN)
- Gaussian Naive Bayes
- Regresión Logística
- Árboles de Decisión
- Support Vector Machines (SVM)
- Random Forest

**Procedimiento:**

1. Divide el dataset en conjunto de **entrenamiento** (80%) y **validación** (20%) usando `train_test_split` con `stratify=y`.
2. Para cada clasificador:
   - Define el espacio de búsqueda de hiperparámetros.
   - Usa **validación cruzada 5-fold** sobre el conjunto de entrenamiento para seleccionar los mejores hiperparámetros.
   - Emplea una estrategia de búsqueda adecuada:
     - **GridSearchCV**: búsqueda exhaustiva (ideal para espacios pequeños).
     - **RandomizedSearchCV**: búsqueda aleatoria (más eficiente con espacios amplios).
     - **Bayesian Optimization** (opcional): para búsquedas más inteligentes, usando librerías como `optuna` o `skopt`.
3. Guarda el mejor modelo encontrado para cada clasificador con su configuración óptima.



In [4]:
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import numpy as np

RANDOM_STATE = 42
CV_FOLDS = 5
SCORING = 'f1_macro'

# Split 80/20 con estratificacion
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Estructuras para guardar resultados del Paso 4
best_models = {}
best_params = {}
cv_best_scores = {}
search_objects = {}

print(f'X_train: {X_train.shape} | X_val: {X_val.shape}')
print(f'y_train: {y_train.shape} | y_val: {y_val.shape}')

X_train: (15949, 520) | X_val: (3988, 520)
y_train: (15949,) | y_val: (3988,)


In [5]:
from sklearn.neighbors import KNeighborsClassifier

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

knn_param_grid = {
    'model__n_neighbors': [3, 5, 7, 9, 11],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan']
}

knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    scoring=SCORING,
    cv=CV_FOLDS,
    n_jobs=-1,
    verbose=1
)

knn_search.fit(X_train, y_train)

best_models['KNN'] = knn_search.best_estimator_
best_params['KNN'] = knn_search.best_params_
cv_best_scores['KNN'] = knn_search.best_score_
search_objects['KNN'] = knn_search

knn_val_pred = knn_search.best_estimator_.predict(X_val)
knn_val_f1 = f1_score(y_val, knn_val_pred, average='macro')

print('KNN listo')
print('Mejores hiperparametros:', knn_search.best_params_)
print(f'Mejor CV ({SCORING}): {knn_search.best_score_:.4f}')
print(f'F1-macro en validacion: {knn_val_f1:.4f}')

Fitting 5 folds for each of 20 candidates, totalling 100 fits
KNN listo
Mejores hiperparametros: {'model__metric': 'manhattan', 'model__n_neighbors': 3, 'model__weights': 'distance'}
Mejor CV (f1_macro): 0.9947
F1-macro en validacion: 0.9968


In [6]:
from sklearn.naive_bayes import GaussianNB

gnb_param_grid = {
    'var_smoothing': np.logspace(-11, -7, 9)
}

gnb_search = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=gnb_param_grid,
    scoring=SCORING,
    cv=CV_FOLDS,
    n_jobs=-1,
    verbose=1
)

gnb_search.fit(X_train, y_train)

best_models['GaussianNB'] = gnb_search.best_estimator_
best_params['GaussianNB'] = gnb_search.best_params_
cv_best_scores['GaussianNB'] = gnb_search.best_score_
search_objects['GaussianNB'] = gnb_search

gnb_val_pred = gnb_search.best_estimator_.predict(X_val)
gnb_val_f1 = f1_score(y_val, gnb_val_pred, average='macro')

print('GaussianNB listo')
print('Mejores hiperparametros:', gnb_search.best_params_)
print(f'Mejor CV ({SCORING}): {gnb_search.best_score_:.4f}')
print(f'F1-macro en validacion: {gnb_val_f1:.4f}')

Fitting 5 folds for each of 9 candidates, totalling 45 fits
GaussianNB listo
Mejores hiperparametros: {'var_smoothing': np.float64(1e-07)}
Mejor CV (f1_macro): 0.5700
F1-macro en validacion: 0.5618


In [7]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

lr_param_grid = {
    'model__C': [0.1, 1.0, 10.0],
    'model__solver': ['lbfgs'],
    'model__class_weight': [None, 'balanced']
}

lr_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    scoring=SCORING,
    cv=CV_FOLDS,
    n_jobs=-1,
    verbose=1
)

lr_search.fit(X_train, y_train)

best_models['LogisticRegression'] = lr_search.best_estimator_
best_params['LogisticRegression'] = lr_search.best_params_
cv_best_scores['LogisticRegression'] = lr_search.best_score_
search_objects['LogisticRegression'] = lr_search

lr_val_pred = lr_search.best_estimator_.predict(X_val)
lr_val_f1 = f1_score(y_val, lr_val_pred, average='macro')

print('Logistic Regression listo')
print('Mejores hiperparametros:', lr_search.best_params_)
print(f'Mejor CV ({SCORING}): {lr_search.best_score_:.4f}')
print(f'F1-macro en validacion: {lr_val_f1:.4f}')

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Logistic Regression listo
Mejores hiperparametros: {'model__C': 1.0, 'model__class_weight': 'balanced', 'model__solver': 'lbfgs'}
Mejor CV (f1_macro): 0.9934
F1-macro en validacion: 0.9927


In [8]:
from sklearn.tree import DecisionTreeClassifier

dt_param_dist = {
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

dt_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_distributions=dt_param_dist,
    n_iter=12,
    scoring=SCORING,
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

dt_search.fit(X_train, y_train)

best_models['DecisionTree'] = dt_search.best_estimator_
best_params['DecisionTree'] = dt_search.best_params_
cv_best_scores['DecisionTree'] = dt_search.best_score_
search_objects['DecisionTree'] = dt_search

dt_val_pred = dt_search.best_estimator_.predict(X_val)
dt_val_f1 = f1_score(y_val, dt_val_pred, average='macro')

print('Decision Tree listo')
print('Mejores hiperparametros:', dt_search.best_params_)
print(f'Mejor CV ({SCORING}): {dt_search.best_score_:.4f}')
print(f'F1-macro en validacion: {dt_val_f1:.4f}')

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Decision Tree listo
Mejores hiperparametros: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'gini'}
Mejor CV (f1_macro): 0.9679
F1-macro en validacion: 0.9738


In [9]:
from sklearn.svm import SVC

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC())
])

svm_param_dist = {
    'model__C': [0.1, 1, 10, 30],
    'model__gamma': ['scale', 'auto', 0.01, 0.001],
    'model__kernel': ['rbf', 'poly']
}

svm_search = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=svm_param_dist,
    n_iter=8,
    scoring=SCORING,
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

svm_search.fit(X_train, y_train)

best_models['SVM'] = svm_search.best_estimator_
best_params['SVM'] = svm_search.best_params_
cv_best_scores['SVM'] = svm_search.best_score_
search_objects['SVM'] = svm_search

svm_val_pred = svm_search.best_estimator_.predict(X_val)
svm_val_f1 = f1_score(y_val, svm_val_pred, average='macro')

print('SVM listo')
print('Mejores hiperparametros:', svm_search.best_params_)
print(f'Mejor CV ({SCORING}): {svm_search.best_score_:.4f}')
print(f'F1-macro en validacion: {svm_val_f1:.4f}')

Fitting 5 folds for each of 8 candidates, totalling 40 fits
SVM listo
Mejores hiperparametros: {'model__kernel': 'rbf', 'model__gamma': 0.001, 'model__C': 30}
Mejor CV (f1_macro): 0.9939
F1-macro en validacion: 0.9950


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_param_dist = {
    'n_estimators': [150, 250, 350],
    'max_depth': [None, 20, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=10,
    scoring=SCORING,
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

best_models['RandomForest'] = rf_search.best_estimator_
best_params['RandomForest'] = rf_search.best_params_
cv_best_scores['RandomForest'] = rf_search.best_score_
search_objects['RandomForest'] = rf_search

rf_val_pred = rf_search.best_estimator_.predict(X_val)
rf_val_f1 = f1_score(y_val, rf_val_pred, average='macro')

print('Random Forest listo')
print('Mejores hiperparametros:', rf_search.best_params_)
print(f'Mejor CV ({SCORING}): {rf_search.best_score_:.4f}')
print(f'F1-macro en validacion: {rf_val_f1:.4f}')

print('\nModelos optimizados hasta ahora:')
print(list(best_models.keys()))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Random Forest listo
Mejores hiperparametros: {'n_estimators': 350, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
Mejor CV (f1_macro): 0.9967
F1-macro en validacion: 0.9972

Modelos optimizados hasta ahora:
['KNN', 'GaussianNB', 'LogisticRegression', 'DecisionTree', 'SVM', 'RandomForest']


---

## Paso 5: Crear una tabla resumen de los mejores modelos

**Instrucciones:**

Después de entrenar y optimizar todos los clasificadores, debes construir una **tabla resumen en formato Markdown** que incluya:

- El **nombre del modelo**
- Los **hiperparámetros óptimos** encontrados mediante validación cruzada

### Requisitos:

- La tabla debe estar escrita en formato **Markdown**.
- Cada fila debe corresponder a uno de los modelos evaluados.
- Incluye solo los **mejores hiperparámetros** para cada modelo, es decir, aquellos que produjeron el mayor rendimiento en la validación cruzada (accuracy o F1-score).
- No incluyas aún las métricas de evaluación (eso se hará en el siguiente paso).

### Ejemplo de formato:


| Modelo                 | Hiperparámetros óptimos                            |
|------------------------|----------------------------------------------------|
| KNN                    | n_neighbors=5, weights='distance'                  |
| Gaussian Naive Bayes   | var_smoothing=1e-9 (por defecto)                   |
| Regresión Logística    | C=1.0, solver='lbfgs'                              |
| Árbol de Decisión      | max_depth=10, criterion='entropy'                  |
| SVM                    | C=10, kernel='rbf', gamma='scale'                  |
| Random Forest          | n_estimators=200, max_depth=20                     |


In [11]:
from IPython.display import Markdown, display

# Paso 5: construir tabla en formato Markdown usando los mejores hiperparámetros del Paso 4
model_order = ['KNN', 'GaussianNB', 'LogisticRegression', 'DecisionTree', 'SVM', 'RandomForest']
model_names = {
    'KNN': 'K-Nearest Neighbors (KNN)',
    'GaussianNB': 'Gaussian Naive Bayes',
    'LogisticRegression': 'Regresión Logística',
    'DecisionTree': 'Árbol de Decisión',
    'SVM': 'Support Vector Machine (SVM)',
    'RandomForest': 'Random Forest'
}

if 'best_params' not in globals() or len(best_params) == 0:
    raise RuntimeError('No existe best_params. Ejecuta primero todas las celdas del Paso 4.')

lines = [
    '| Modelo | Hiperparámetros óptimos |',
    '|---|---|'
 ]

for key in model_order:
    params = best_params.get(key, {})
    if params:
        params_text = ', '.join([f"{k}={v}" for k, v in params.items()])
    else:
        params_text = 'No disponible (modelo no ejecutado)'
    lines.append(f"| {model_names[key]} | {params_text} |")

tabla_paso5_md = '\n'.join(lines)
print(tabla_paso5_md)

display(Markdown('### Tabla resumen de mejores modelos'))
display(Markdown(tabla_paso5_md))

| Modelo | Hiperparámetros óptimos |
|---|---|
| K-Nearest Neighbors (KNN) | model__metric=manhattan, model__n_neighbors=3, model__weights=distance |
| Gaussian Naive Bayes | var_smoothing=1e-07 |
| Regresión Logística | model__C=1.0, model__class_weight=balanced, model__solver=lbfgs |
| Árbol de Decisión | min_samples_split=2, min_samples_leaf=1, max_depth=None, criterion=gini |
| Support Vector Machine (SVM) | model__kernel=rbf, model__gamma=0.001, model__C=30 |
| Random Forest | n_estimators=350, min_samples_split=2, min_samples_leaf=1, max_features=log2, max_depth=None |


### Tabla resumen de mejores modelos

| Modelo | Hiperparámetros óptimos |
|---|---|
| K-Nearest Neighbors (KNN) | model__metric=manhattan, model__n_neighbors=3, model__weights=distance |
| Gaussian Naive Bayes | var_smoothing=1e-07 |
| Regresión Logística | model__C=1.0, model__class_weight=balanced, model__solver=lbfgs |
| Árbol de Decisión | min_samples_split=2, min_samples_leaf=1, max_depth=None, criterion=gini |
| Support Vector Machine (SVM) | model__kernel=rbf, model__gamma=0.001, model__C=30 |
| Random Forest | n_estimators=350, min_samples_split=2, min_samples_leaf=1, max_features=log2, max_depth=None |

### Tabla resumen de mejores modelos

| Modelo | Hiperparámetros óptimos |
|---|---|
| K-Nearest Neighbors (KNN) | model__metric=manhattan, model__n_neighbors=3, model__weights=distance |
| Gaussian Naive Bayes | var_smoothing=1e-07 |
| Regresión Logística | model__C=1.0, model__class_weight=balanced, model__solver=lbfgs |
| Árbol de Decisión | min_samples_split=2, min_samples_leaf=1, max_depth=None, criterion=gini |
| Support Vector Machine (SVM) | model__kernel=rbf, model__gamma=0.001, model__C=30 |
| Random Forest | n_estimators=350, min_samples_split=2, min_samples_leaf=1, max_features=log2, max_depth=None |

---

## Paso 6: Preparar los datos finales para evaluación

**Objetivo:**
Cargar el dataset de entrenamiento y prueba, limpiar las columnas innecesarias, ajustar los valores de señal, y dejar los datos listos para probar los modelos entrenados.

**Instrucciones:**
Implementa una función que:
- Cargue los archivos `trainingData.csv` y `validationData.csv`
- Elimine las columnas irrelevantes (`LONGITUDE`, `LATITUDE`, `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP`)
- Reemplace los valores `100` por `-100` en las columnas `WAP001` a `WAP520`
- Separe las características (`X`) y la variable objetivo (`FLOOR`)
- Devuelva los conjuntos `X_train`, `X_test`, `y_train`, `y_test`

In [12]:
def load_and_prepare_final_data(
    train_path='dataset/trainingData.csv',
    test_path='dataset/validationData.csv'
 ):
    import pandas as pd

    cols_to_drop = [
        'LONGITUDE', 'LATITUDE', 'SPACEID', 'RELATIVEPOSITION',
        'USERID', 'PHONEID', 'TIMESTAMP'
    ]

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df = train_df.drop(columns=cols_to_drop, errors='ignore').copy()
    test_df = test_df.drop(columns=cols_to_drop, errors='ignore').copy()

    if 'FLOOR' not in train_df.columns or 'FLOOR' not in test_df.columns:
        raise KeyError("La columna 'FLOOR' debe existir en trainingData.csv y validationData.csv")

    wap_cols_train = [c for c in train_df.columns if c.startswith('WAP')]
    wap_cols_test = [c for c in test_df.columns if c.startswith('WAP')]

    if set(wap_cols_train) != set(wap_cols_test):
        raise ValueError('Las columnas WAP no coinciden entre train y test.')

    # Asegurar mismo orden de columnas en train y test
    wap_cols = sorted(wap_cols_train)

    X_train = train_df[wap_cols].copy()
    y_train = train_df['FLOOR'].copy()
    X_test = test_df[wap_cols].copy()
    y_test = test_df['FLOOR'].copy()

    # Reemplazar código de ausencia de señal
    X_train[X_train == 100] = -100
    X_test[X_test == 100] = -100

    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = load_and_prepare_final_data()

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_test: {X_test.shape} | y_test: {y_test.shape}')
print(f'Clases en y_train: {sorted(y_train.unique())}')
print(f'Clases en y_test: {sorted(y_test.unique())}')

X_train: (19937, 520) | y_train: (19937,)
X_test: (1111, 520) | y_test: (1111,)
Clases en y_train: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Clases en y_test: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


---

## Paso 7: Evaluar modelos optimizados en el conjunto de prueba

**Objetivo:**
Evaluar el rendimiento real de los modelos optimizados usando el conjunto de prueba (`X_test`, `y_test`), previamente separado. Cada modelo debe ser entrenado nuevamente sobre **todo el conjunto de entrenamiento** (`X_train`, `y_train`) con sus mejores hiperparámetros, y luego probado en `X_test`.

**Instrucciones:**

1. Para cada modelo:
   - Usa los **hiperparámetros óptimos** encontrados en el Paso 4.
   - Entrena el modelo con `X_train` y `y_train`.
   - Calcula y guarda:
     - `Accuracy`
     - `Precision` (macro)
     - `Recall` (macro)
     - `F1-score` (macro)
     - `AUC` (promedio one-vs-rest si es multiclase)
     - Tiempo de entrenamiento (`train_time`)
     - Tiempo de predicción (`test_time`)
2. Muestra todos los resultados en una **tabla comparativa**


In [13]:
import time
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

if 'best_params' not in globals() or len(best_params) == 0:
    raise RuntimeError('No se encontró best_params. Ejecuta primero todo el Paso 4.')

def strip_model_prefix(params_dict):
    return {k.replace('model__', ''): v for k, v in params_dict.items()}

# Reconstruir modelos con hiperparámetros óptimos del Paso 4
models_for_test = {
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(**strip_model_prefix(best_params['KNN'])))
    ]),
    'GaussianNB': GaussianNB(**best_params['GaussianNB']),
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=2000, random_state=RANDOM_STATE,
            **strip_model_prefix(best_params['LogisticRegression'])
        ))
    ]),
    'DecisionTree': DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        **best_params['DecisionTree']
    ),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(**strip_model_prefix(best_params['SVM'])))
    ]),
    'RandomForest': RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **best_params['RandomForest']
    )
}

# Etiquetas para AUC multiclase OVR
classes_sorted = np.sort(y_train.unique())
y_test_bin = label_binarize(y_test, classes=classes_sorted)

results = []

for model_name, model in models_for_test.items():
    # Entrenamiento
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    # Predicción
    t1 = time.perf_counter()
    y_pred = model.predict(X_test)
    test_time = time.perf_counter() - t1

    # Métricas principales
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # AUC multiclase (OVR) usando predict_proba o decision_function
    try:
        if hasattr(model, 'predict_proba'):
            y_score = model.predict_proba(X_test)
        elif hasattr(model, 'decision_function'):
            y_score = model.decision_function(X_test)
        else:
            y_score = None

        if y_score is not None:
            if isinstance(y_score, list):
                y_score = np.asarray(y_score)
            if getattr(y_score, 'ndim', 2) == 1:
                y_score = y_score.reshape(-1, 1)
            auc = roc_auc_score(y_test_bin, y_score, multi_class='ovr', average='macro')
        else:
            auc = np.nan
    except Exception:
        auc = np.nan

    results.append({
        'Modelo': model_name,
        'Accuracy': acc,
        'Precision_macro': prec,
        'Recall_macro': rec,
        'F1_macro': f1,
        'AUC_ovr_macro': auc,
        'train_time_s': train_time,
        'test_time_s': test_time
    })

results_df = pd.DataFrame(results).sort_values(by='F1_macro', ascending=False).reset_index(drop=True)

# Redondear para visualización
results_df_display = results_df.copy()
for col in ['Accuracy', 'Precision_macro', 'Recall_macro', 'F1_macro', 'AUC_ovr_macro', 'train_time_s', 'test_time_s']:
    results_df_display[col] = results_df_display[col].round(4)

display(results_df_display)

,Modelo,Accuracy,Precision_macro,Recall_macro,F1_macro,AUC_ovr_macro,train_time_s,test_time_s
0,RandomForest,0.9118,0.9252,0.8820,0.8983,0.9852,3.2790,0.1866
1,LogisticRegression,0.8893,0.8659,0.8983,0.8795,0.9729,18.0220,0.0131
2,KNN,0.8641,0.8649,0.8553,0.8547,0.9334,0.4025,3.5493
3,SVM,0.8308,0.8328,0.8289,0.8228,0.9423,21.5892,1.9282
4,DecisionTree,0.8128,0.8193,0.8001,0.8062,0.8766,2.0479,0.0078
5,GaussianNB,0.4914,0.4864,0.5984,0.4725,0.7923,0.3135,0.0534


---
## Paso 8: Selección y justificación del mejor modelo

**Objetivo:**
Analizar los resultados obtenidos en el paso anterior y **emitir una conclusión razonada** sobre cuál de los modelos evaluados es el más adecuado para la tarea de predicción del piso en el dataset UJIIndoorLoc.

**Instrucciones:**

- Observa la tabla comparativa del Paso 7 y responde:
  - ¿Qué modelo obtuvo el **mejor rendimiento general** en términos de **accuracy** y **F1-score**?
  - ¿Qué tan consistente fue su rendimiento en **precision** y **recall**?
  - ¿Tiene un **tiempo de entrenamiento o inferencia** excesivamente alto?
  - ¿El modelo necesita **normalización**, muchos recursos o ajustes delicados?
- Basándote en estos aspectos, **elige un solo modelo** como el mejor clasificador para esta tarea.
- **Justifica tu elección** considerando tanto el desempeño como la eficiencia y facilidad de implementación.


### Selección del mejor modelo

Con base en la tabla comparativa del Paso 7, el modelo con **mejor rendimiento general** es **Random Forest**.

- **Mejor Accuracy**: 0.9118 (el más alto de todos).
- **Mejor F1-score macro**: 0.8983 (también el más alto).
- **Mejor AUC OVR macro**: 0.9852, lo que indica muy buena capacidad de discriminación multiclase.

### Consistencia en Precision y Recall

Random Forest mantiene métricas equilibradas:

- **Precision macro**: 0.9252
- **Recall macro**: 0.8820

Esto muestra buen desempeño global entre clases, sin una caída fuerte en recall respecto a precisión.

### Coste computacional (entrenamiento e inferencia)

- **Random Forest**: `train_time = 3.2790 s`, `test_time = 0.1866 s`.
- **Logistic Regression** tiene inferencia muy rápida (`0.0131 s`), pero menor accuracy/F1 que Random Forest.
- **KNN** y **SVM** presentan inferencia más costosa (`3.5493 s` y `1.9282 s`, respectivamente), lo cual puede ser menos conveniente en producción.

### Requerimientos prácticos

- KNN, SVM y Regresión Logística dependen más de normalización adecuada y ajuste fino de hiperparámetros.
- Random Forest es más robusto al escalado de variables y suele requerir menos sensibilidad al preprocesamiento.

### Conclusión final

**El mejor clasificador para esta tarea es Random Forest**, porque ofrece el mejor balance entre desempeño predictivo (accuracy, F1, AUC), consistencia entre clases y costos computacionales razonables para entrenamiento e inferencia.

---

## Rúbrica de Evaluación

| Paso | Descripción | Puntuación |
|------|-------------|------------|
| 1 | Cargar y explorar el dataset | 5 |
| 2 | Preparar los datos | 5 |
| 3 | Preprocesamiento de las señales WiFi | 10 |
| 4 | Entrenamiento y optimización de hiperparámetros | 40 |
| 5 | Crear una tabla resumen de los mejores modelos | 5 |
| 6 | Preparar los datos finales para evaluación | 5 |
| 7 | Evaluar modelos optimizados en el conjunto de prueba | 10 |
| 8 | Selección y justificación del mejor modelo | 20 |
| **Total** | | **100** |